# זכותי: אבן דרך 1 (כלים ומצב)

המחברת רק מריצה את הקוד שנמצא בקבצי `src/` ו-`tests/` במאגר. כברירת מחדל היא רצה עם **רכיב החלטה מדומה, בלי מפתח ובלי עלות**. כל הנתונים סינתטיים.

## 1. שכפול המאגר והתקנה
יש להחליף את `REPO_URL` בקישור למאגר שלכם.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/Nikvai/zchuti-milestone1"   # TODO: replace with the submitted repository
REPO_DIR = "zchuti"

if not os.path.exists("src/agent.py"):               # skip cloning when already inside the repo
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print("working dir:", os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. הבדיקות (פקודה אחת, מודל מדומה, בלי מפתח)

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-1500:])
assert result.returncode == 0, "tests failed"

## 3. הדגמת הסוכן
המפתח (אם קיים) נקרא מ-Colab Secrets בשם `ANTHROPIC_API_KEY`. בלעדיו רץ המודל המדומה.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass                                              # no Colab or no secret: the mock decision component is used

from src.agent import Trace, build_agent, make_decider

decider = make_decider()
print("decision component:", decider.name)
agent = build_agent(decider=decider)
trace = Trace()
for message in ["גיל 26, סטודנט", "4000", "כן"]:
    reply = agent.handle_message("demo", message, trace)
    print(f"משתמש: {message}\nסוכן ({reply['status']}): {reply['text']}\n")

## 4. כשל זמני (כלי מדומה שנכשל פעמיים ואז מצליח)

In [ ]:
from src.mocks import FlakyLoader

flaky_agent = build_agent(loader=FlakyLoader(failures=2))
flaky_trace = Trace()
reply = flaky_agent.handle_message("flaky", "גיל 26, סטודנט, מועסק, הכנסה 4,000", flaky_trace)
first_tool_result = [e for e in flaky_trace.events if e["event"] == "tool_result"][0]["result"]
print("status:", reply["status"], "| attempts for the first tool call:", first_tool_result["attempts"])

## 5. הערכה והשוואת הקשר

In [ ]:
import subprocess, sys
for module in ("src.evaluate", "src.compare_context"):
    print(subprocess.run([sys.executable, "-m", module], capture_output=True, text=True).stdout)